# &#x2B21; J.A.R.V.I.S. &middot; VoxCPM2 Hindi / Hinglish TTS Engine &middot; v3
### *Readlyte Production &middot; OpenBMB VoxCPM2 &middot; 2B Params &middot; 48 kHz &middot; 30 Languages*

---
> **Run all cells top-to-bottom (Runtime &rarr; Run all).**
> Set Runtime Type to **T4 GPU** before starting.
> The **only** lines you need to edit are:
> - `ENABLE_NORMALIZATION` in **Cell 3**
> - The `USER CONFIG` block in **Cell 5**

| Cell | Purpose |
|------|---------|
| 1 | GPU / Environment Diagnostics |
| 2 | Install Dependencies |
| 3 | Upload Text + Preprocessing (**normalization toggle here**) |
| 4 | Upload Reference Audio (optional voice clone) |
| 5 | Edit TTS Config Here |
| 6 | Load Model, Generate & Stitch TTS |
| 7 | Playback & Download |

---

### What is new in v3

| Feature | Detail |
|---------|--------|
| **Normalization toggle** | `ENABLE_NORMALIZATION` in Cell 3. Layer 1 (NFC, ZW-char cleanup) always runs. Layer 2 (curly quotes, en-dash, ellipsis, punctuation spacing) is now opt-in. |
| **Chapter heading detector** | Lines matching `Chapter N`, `अध्याय N`, `=====`, `## Heading`, etc. get a dedicated 2 s pause — no manual marking needed. |
| **Min-chunk enforcement** | Orphan short chunks (< `MIN_CHUNK_SIZE` chars) are merged with their neighbour so the model never receives a fragment too short to produce natural prosody. |
| **Terminal-punctuation guard** | Every chunk sent to TTS ends with `।` or `.` — the model always generates a natural phrase-final cadence rather than cutting mid-breath. |
| **Crossfade stitcher** | Short configurable fade-out (`CROSSFADE_MS`, default 15 ms) applied to each chunk tail before the silence gap — eliminates click / discontinuity artifacts at stitch boundaries. |
| **Resume / checkpoint** | Per-chunk `.npy` cache in `CHUNKS_DIR`. On re-run after a crash only failed chunks are regenerated; completed chunks load from disk instantly. |
| **3-attempt retry** | Was 2. Handles transient VRAM spikes and model hiccups more reliably. |
| **Peak normalisation** | Output peaks at -1 dBFS (was -0.5 dB). |
| **Hinglish style prefix** | Tuned for modern Hindi web-series / novel narration — warm, expressive, conversational tone, no foul language. |


In [ ]:
# ================================================================
# CELL 1 — SYSTEM DIAGNOSTICS  ·  JARVIS BOOT SEQUENCE  v3
# ================================================================
import subprocess, sys, os, platform
from IPython.display import display, HTML

# ── Iron Man Dashboard Helpers (available to ALL cells) ──────────

def _esc(s):
    return str(s).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def _panel(title, rows, accent="#e74c3c", note="", width="700px"):
    inner = ""
    for row in rows:
        if row is None:
            inner += '<tr><td colspan="3"><div style="border-top:1px solid #1e1e1e;margin:5px 0"></div></td></tr>'
            continue
        if isinstance(row, str):
            inner += (f'<tr><td colspan="3" style="color:{accent};font-size:10px;letter-spacing:2px;'
                      f'padding:9px 0 3px;text-transform:uppercase;font-weight:bold">{_esc(row)}</td></tr>')
            continue
        label, value, *rest = row
        status = rest[0] if rest else ""
        sc = ("#2ecc71" if any(x in status for x in ("✅","OK","ONLINE","NOMINAL","ACTIVE"))
              else "#f39c12" if any(x in status for x in ("⚠","WARN","LOW","PENDING"))
              else "#e74c3c" if any(x in status for x in ("❌","FAIL","OFFLINE","ERROR"))
              else "#666")
        inner += (
            f'<tr>'
            f'<td style="color:#555;font-size:10px;letter-spacing:1.5px;text-transform:uppercase;'
            f'padding:4px 14px 4px 0;white-space:nowrap;vertical-align:top">{_esc(label)}</td>'
            f'<td style="color:#ddd;font-size:12px;padding:4px 8px 4px 0;'
            f'font-family:Courier New,monospace;word-break:break-word">{_esc(value)}</td>'
            f'<td style="color:{sc};font-size:11px;padding:4px 0;white-space:nowrap;'
            f'vertical-align:top">{_esc(status)}</td>'
            f'</tr>'
        )
    note_html = (f'<div style="color:#444;font-size:10px;letter-spacing:.5px;margin-top:10px;'
                 f'border-top:1px solid #1a1a1a;padding-top:8px">{_esc(note)}</div>' if note else "")
    return (
        f'<div style="background:#060606;border:1.5px solid {accent};border-radius:9px;'
        f'padding:18px 22px;margin:10px 0;font-family:Courier New,monospace;'
        f'max-width:{width};box-shadow:0 0 18px {accent}1a">'
        f'<div style="color:{accent};font-size:13px;font-weight:bold;letter-spacing:3px;'
        f'border-bottom:1px solid #1a1a1a;padding-bottom:9px;margin-bottom:12px">'
        f'&#x2B21; {_esc(title)}</div>'
        f'<table style="border-collapse:collapse;width:100%">{inner}</table>'
        f'{note_html}</div>'
    )

def _banner(l1, l2="", l3=""):
    sub2 = (f'<div style="color:#f39c12;font-size:10px;letter-spacing:3px;margin-top:4px">'
            f'{_esc(l2)}</div>') if l2 else ""
    sub3 = (f'<div style="color:#555;font-size:10px;letter-spacing:2px;margin-top:3px">'
            f'{_esc(l3)}</div>') if l3 else ""
    return (
        '<div style="background:linear-gradient(135deg,#070707 0%,#1b0303 100%);'
        'border:2px solid #c0392b;border-radius:12px;padding:20px 26px;margin:10px 0;'
        'font-family:Courier New,monospace;box-shadow:0 0 28px #c0392b33,inset 0 0 80px #0f000011">'
        '<table style="border-collapse:collapse;width:100%"><tr>'
        '<td style="width:58px;vertical-align:middle">'
        '<div style="color:#e74c3c;font-size:50px;text-shadow:0 0 18px #e74c3c99;line-height:1">&#x2B21;</div></td>'
        f'<td style="vertical-align:middle;padding-left:16px">'
        f'<div style="color:#e74c3c;font-size:22px;font-weight:bold;letter-spacing:5px;'
        f'text-shadow:0 0 10px #e74c3c55">{_esc(l1)}</div>'
        f'{sub2}{sub3}</td>'
        '<td style="text-align:right;vertical-align:top">'
        '<div style="color:#222;font-size:9px;letter-spacing:1px;line-height:1.8">'
        'READLYTE<br>PROD-SYS<br>v3.0</div></td>'
        '</tr></table></div>'
    )

def _alert(msg, level="info"):
    c  = {"info":"#3498db","success":"#2ecc71","warn":"#f39c12","error":"#e74c3c"}.get(level,"#888")
    ic = {"info":"i","success":"OK","warn":"!","error":"X"}.get(level,"*")
    emoji = {"info":"ℹ️","success":"✅","warn":"⚠️","error":"❌"}.get(level,"•")
    return (f'<div style="background:{c}14;border-left:3px solid {c};padding:9px 14px;'
            f'margin:6px 0;font-family:Courier New,monospace;font-size:12px;'
            f'color:#ccc;border-radius:0 6px 6px 0;max-width:700px">'
            f'{emoji} {_esc(msg)}</div>')

def _pbar(pct, label="", color="#e74c3c"):
    p   = max(0, min(100, pct))
    lbl = (f'<div style="color:#666;font-size:10px;letter-spacing:1px;margin-bottom:3px;'
           f'text-transform:uppercase">{_esc(label)}</div>') if label else ""
    return (f'{lbl}<div style="background:#111;border-radius:3px;height:5px;'
            f'max-width:700px;overflow:hidden">'
            f'<div style="background:linear-gradient(90deg,{color}99,{color});'
            f'width:{p}%;height:100%;border-radius:3px"></div></div>'
            f'<div style="color:{color};font-size:9px;text-align:right;'
            f'max-width:700px;margin-top:1px">{p:.1f}%</div>')

# ── DIAGNOSTICS ──────────────────────────────────────────────────
display(HTML(_banner("J.A.R.V.I.S.",
                     "VoxCPM2 HINDI/HINGLISH TTS ENGINE  READLYTE",
                     "CELL 1 / 7  BOOT SEQUENCE")))
rows   = []
gpu_ok = False

rows.append("RUNTIME")
rows.append(("Python",   sys.version.split()[0], "ACTIVE"))
rows.append(("Platform", f"{platform.system()} {platform.release()}", ""))
rows.append(None)
rows.append("GPU STATUS")
try:
    raw = subprocess.check_output(
        ["nvidia-smi","--query-gpu=name,memory.total,memory.free,driver_version",
         "--format=csv,noheader"],
        encoding="utf-8").strip().split(",")
    gn, vt, vf, dr = [x.strip() for x in raw[:4]]
    vmb    = int("".join(c for c in vt if c.isdigit()))
    gpu_ok = vmb >= 7000
    rows  += [("Device",     gn,  "ONLINE"),
              ("VRAM Total", vt,  "✅ OK" if gpu_ok else "⚠ LOW (<7 GB)"),
              ("VRAM Free",  vf,  ""),
              ("Driver",     dr,  "")]
except Exception as ex:
    rows.append(("GPU", f"Not detected -- {ex}", "❌ OFFLINE"))

rows.append(None)
rows.append("PYTORCH / CUDA")
try:
    import torch
    cuda = torch.cuda.is_available()
    rows += [("PyTorch", torch.__version__, ""),
             ("CUDA", torch.version.cuda if cuda else "N/A", "✅ OK" if cuda else "❌ NO CUDA"),
             ("GPU",  torch.cuda.get_device_name(0) if cuda else "--", "")]
except ImportError:
    rows.append(("PyTorch", "Not installed -- Cell 2 will install it", "⚠ PENDING"))

rows.append(None)
rows.append("STORAGE")
try:
    dk       = subprocess.check_output(["df","-h","/"],encoding="utf-8").strip().split("\n")[-1].split()
    used_pct = int(dk[4].replace("%",""))
    rows.append(("Disk (/)", f"{dk[3]} free / {dk[1]} total ({dk[4]} used)",
                 "✅ OK" if used_pct < 85 else "⚠ LOW SPACE"))
except Exception:
    rows.append(("Disk", "Unable to read", ""))

display(HTML(_panel("SYSTEM DIAGNOSTICS", rows,
    note="VoxCPM2 requires >= 8 GB VRAM and ~15 GB disk space on first run (model download).")))
if not gpu_ok:
    display(HTML(_alert("T4 GPU REQUIRED -- Runtime > Change Runtime Type > T4 GPU > Save", "warn")))
else:
    display(HTML(_alert("All systems nominal. Proceed to Cell 2.", "success")))


In [ ]:
# ================================================================
# CELL 2 — INSTALL DEPENDENCIES
# ================================================================
import subprocess, sys, time
from IPython.display import display, HTML, clear_output

display(HTML(_banner("INSTALLING DEPENDENCIES", "VoxCPM2 + AUDIO STACK", "CELL 2 / 7")))

PACKAGES = [
    ("voxcpm",          "VoxCPM2 core TTS library",       True),
    ("transformers",    "HuggingFace model loader",        False),
    ("accelerate",      "Multi-GPU / CPU offload helper",  False),
    ("huggingface_hub", "HF Hub download utility",         False),
    ("soundfile",       "Audio file read / write",         False),
    ("scipy",           "Signal processing",               False),
    ("librosa",         "Audio analysis utilities",        False),
    ("numpy",           "Numerical arrays",                False),
    ("tqdm",            "Progress bars",                   False),
    ("ipywidgets",      "Colab UI widgets",                False),
]

rows   = []
all_ok = True
for i, (pkg, desc, upgrade) in enumerate(PACKAGES, 1):
    t0     = time.time()
    flags  = ["--upgrade"] if upgrade else []
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"] + flags,
        capture_output=True, text=True)
    elapsed = time.time() - t0
    ok = result.returncode == 0
    if not ok:
        all_ok = False
    rows.append((
        f"[{i:02d}/{len(PACKAGES):02d}] {pkg}",
        f"{desc}  ({elapsed:.1f}s)",
        "✅ OK" if ok else "❌ FAILED"
    ))

display(HTML(_panel("PACKAGE INSTALLATION", rows,
    note="If voxcpm fails: !pip install git+https://github.com/openbmb/voxcpm.git")))

verify = [("voxcpm", None), ("soundfile", None), ("numpy", None),
          ("torch", None), ("scipy", None)]
vrows  = []
for mod, _ in verify:
    try:
        m   = __import__(mod)
        ver = getattr(m, "__version__", "ok")
        vrows.append((mod, f"v{ver}", "✅ IMPORTED"))
    except ImportError as e:
        vrows.append((mod, str(e), "❌ FAILED"))
        all_ok = False

display(HTML(_panel("IMPORT VERIFICATION", vrows)))
display(HTML(
    _alert("All packages installed and verified. Proceed to Cell 3.", "success") if all_ok
    else _alert("Some imports failed -- check errors above before continuing.", "error")
))


In [ ]:
# ================================================================
# CELL 3 — UPLOAD TEXT  +  PREPROCESSING
# ================================================================
#
# ┌─────────────────────────────────────────────────────────────────┐
# │  NORMALIZATION TOGGLE                                           │
# │                                                                 │
# │  True  = Layer 2 normalization ON  (recommended default)       │
# │          Converts curly quotes, en-dash, ellipsis, fixes       │
# │          spacing around punctuation, caps repeated marks.       │
# │                                                                 │
# │  False = Layer 2 normalization OFF                             │
# │          Use when your text is already editor-clean /          │
# │          manually punctuated -- raw characters preserved.      │
# └─────────────────────────────────────────────────────────────────┘

# @title Text Input and Normalization
ENABLE_NORMALIZATION = True # @param {type:"boolean"}

# ================================================================

from google.colab import files
from IPython.display import display, HTML
import re, unicodedata, os

display(HTML(_banner(
    "UPLOAD TEXT FILE",
    "HINDI  HINGLISH  DEVANAGARI / LATIN / MIXED",
    "CELL 3 / 7")))
display(HTML(_alert(
    "Upload a UTF-8 .txt file -- Devanagari, Hinglish (Latin), or mixed script supported.",
    "info")))

# ── PREPROCESSING LAYER 1: always-on safe structural cleanup ─────

def _clean_text(text):
    # ── Remove purely decorative separator lines (=====, -----) ──────────────
    # These carry NO spoken content. Removing them turns them into blank lines
    # which become paragraph boundaries (→ natural pause in audio).
    # Must happen BEFORE whitespace-collapse to correctly split mixed
    # paragraphs like "TITLE\n======..." into two separate paragraphs.
    text = text.replace("\r\n", "\n").replace("\r", "\n")   # unify first
    text = re.sub(r"(?m)^[ \t]*[=\-]{5,}[ \t]*$", "", text)
    # Compose Devanagari vowel marks correctly (NFC)
    text = unicodedata.normalize("NFC", text)
    # (line endings already unified above)
    # Remove zero-width / invisible control chars and soft-hyphen
    text = re.sub(r"[\u00ad\u200b\u200c\u200d\u2060\ufeff]", "", text)
    # Collapse consecutive spaces / tabs (NOT newlines)
    text = re.sub(r"[ \t]+", " ", text)
    # Max 2 consecutive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Strip trailing whitespace per line
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    return text.strip()

# ── PREPROCESSING LAYER 2: optional typographic normalization ────

def _normalize_text(text):
    # Curly / smart quotes and angle guillemets -> straight ASCII
    replacements = [
        ("\u2018","'"), ("\u2019","'"), ("\u201a","'"), ("\u201b","'"),
        ("\u201c",'"'), ("\u201d",'"'), ("\u201e",'"'), ("\u201f",'"'),
        ("\u2039","'"), ("\u203a","'"), ("\u00ab",'"'), ("\u00bb",'"'),
    ]
    for src, dst in replacements:
        text = text.replace(src, dst)
    # En-dash -> hyphen; em-dash stays (narrative pause)
    text = text.replace("\u2013", "-")
    # Ellipsis character -> three ASCII dots
    text = text.replace("\u2026", "...")
    # 3+ consecutive dashes -> em-dash
    text = re.sub(r"-{3,}", "\u2014", text)
    # Remove stray space before punctuation
    text = re.sub(r" +([।.!?,;:])", r"\1", text)
    # Ensure exactly one space after sentence-ending marks before a letter
    text = re.sub(r"([।.!?])([^\s\n।.!?])", r"\1 \2", text)
    # Cap run of repeated terminal punctuation (more than 2)
    text = re.sub(r"([।!?]){3,}", r"\1\1", text)
    text = re.sub(r"\.{4,}", "...", text)
    # Narrow no-break space -> regular space
    text = re.sub(r"\u202f", " ", text)
    return text

def preprocess_for_tts(text, normalize=True):
    text = _clean_text(text)
    if normalize:
        text = _normalize_text(text)
    return text

# ── UPLOAD ───────────────────────────────────────────────────────
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded. Re-run this cell and upload a .txt file.")

fname = list(uploaded.keys())[0]
if not fname.lower().endswith(".txt"):
    raise ValueError(f"Expected a .txt file, got: {fname}")

raw = uploaded[fname]
try:
    raw_text = raw.decode("utf-8")
except UnicodeDecodeError:
    raw_text = raw.decode("utf-8", errors="replace")
    display(HTML(_alert(
        "Some bytes could not be decoded -- replaced with U+FFFD. "
        "Re-save your source file as UTF-8 for cleanest results.", "warn")))

INPUT_TEXT_RAW   = raw_text.strip()
INPUT_TEXT_CLEAN = _clean_text(INPUT_TEXT_RAW)
INPUT_TEXT       = preprocess_for_tts(INPUT_TEXT_RAW, normalize=ENABLE_NORMALIZATION)

# ── Diff stats (sequential matcher, character level) ─────────────
def _count_diffs(a, b):
    from difflib import SequenceMatcher
    sm  = SequenceMatcher(None, a, b, autojunk=False)
    return sum(max(j1-i1, j2-i2) for tag, i1, i2, j1, j2 in sm.get_opcodes()
               if tag != "equal")

changes_clean = _count_diffs(INPUT_TEXT_RAW,   INPUT_TEXT_CLEAN)
changes_norm  = _count_diffs(INPUT_TEXT_CLEAN, INPUT_TEXT) if ENABLE_NORMALIZATION else 0

char_count = len(INPUT_TEXT)
word_count = len(INPUT_TEXT.split())
line_count = INPUT_TEXT.count("\n") + 1
para_count = len([p for p in INPUT_TEXT.split("\n\n") if p.strip()])
est_audio  = word_count / 130   # approx 130 wpm Hindi audiobook narration

rows = [
    ("Filename",       fname,                              ""),
    ("Characters",     f"{char_count:,}",                 ""),
    ("Words",          f"{word_count:,}",                 ""),
    ("Lines",          f"{line_count:,}",                 ""),
    ("Paragraphs",     f"{para_count:,}",                 ""),
    ("Est. Duration",  f"~{est_audio:.1f} min @ 130 wpm",""),
    None,
    "PREPROCESSING PIPELINE",
    ("Layer 1  Clean",
     f"{changes_clean} fix(es) -- NFC, ZW-chars, whitespace", "✅ DONE"),
    ("Layer 2  Normalize",
     "ENABLED" if ENABLE_NORMALIZATION else "DISABLED (raw text kept)",
     "✅ ON" if ENABLE_NORMALIZATION else "⚠ OFF"),
    ("  Norm changes",
     f"{changes_norm} typographic fix(es)" if ENABLE_NORMALIZATION else "skipped", ""),
]
display(HTML(_panel(
    "TEXT FILE STATS", rows,
    note="Set ENABLE_NORMALIZATION = False at top of this cell to preserve raw punctuation."
)))

preview_html = (
    '<div style="background:#0a0a0a;border:1px solid #2a2a2a;border-radius:6px;'
    'padding:12px 16px;margin:8px 0;font-family:Courier New,monospace;font-size:12px;'
    'color:#aaa;max-width:700px;white-space:pre-wrap;max-height:220px;overflow-y:auto;'
    'line-height:1.6">'
    + _esc(INPUT_TEXT[:700])
    + ('<span style="color:#555">...</span>' if len(INPUT_TEXT) > 700 else "")
    + '</div>'
)
display(HTML(
    '<div style="color:#666;font-size:10px;letter-spacing:2px;'
    'text-transform:uppercase;margin-top:12px">'
    'PREVIEW -- FIRST 700 CHARS (post-processed)</div>'))
display(HTML(preview_html))
display(HTML(_alert("Text loaded and pre-processed. Proceed to Cell 4.", "success")))


In [ ]:
# ================================================================
# CELL 4 — REFERENCE AUDIO  (OPTIONAL — SKIP FOR DEFAULT VOICE)
# ================================================================
from google.colab import files
from IPython.display import display, HTML
import os, subprocess
import soundfile as sf
import numpy as np

display(HTML(_banner("REFERENCE AUDIO", "VOICE CLONING CONFIG", "CELL 4 / 7")))

mode_rows = [
    ("Default TTS",    "No reference audio. VoxCPM2 built-in voice. Recommended for most use-cases."),
    ("Voice Clone",    "Reference WAV/MP3/FLAC only -- controllable style cloning."),
    ("Ultimate Clone", "Reference audio + exact transcript -- highest voice fidelity."),
]
display(HTML(_panel(
    "CLONING MODES",
    [(m, d, "") for m, d in mode_rows],
    note="Upload a 10-20 s clean, noise-free single-speaker clip for best cloning results."
)))

print("\n>>> UPLOAD your reference audio file (or press Cancel to skip):")
ref_uploaded = files.upload()

REFERENCE_WAV_PATH = None
PROMPT_WAV_PATH    = None
PROMPT_TEXT        = None

if not ref_uploaded:
    display(HTML(_alert("No reference audio -- Default TTS mode active.", "info")))
else:
    ref_fname = list(ref_uploaded.keys())[0]
    ref_ext   = os.path.splitext(ref_fname)[1].lower()

    if ref_ext not in (".wav", ".mp3", ".flac", ".ogg", ".m4a"):
        raise ValueError(
            f"Unsupported format: {ref_ext}. Please use WAV / MP3 / FLAC / OGG / M4A.")

    ref_save = f"/content/reference_audio{ref_ext}"
    with open(ref_save, "wb") as fh:
        fh.write(ref_uploaded[ref_fname])

    # Convert non-WAV to 16 kHz mono WAV (VoxCPM2 works best with WAV)
    if ref_ext != ".wav":
        wav_out = "/content/reference_audio.wav"
        r = subprocess.run(
            ["ffmpeg", "-y", "-i", ref_save, "-ar", "16000", "-ac", "1", wav_out],
            capture_output=True)
        if r.returncode == 0:
            ref_save = wav_out
            display(HTML(_alert(f"Converted {ref_ext} -> WAV 16 kHz mono.", "success")))
        else:
            display(HTML(_alert(
                "ffmpeg conversion failed -- using original file. "
                "Install ffmpeg or upload a .wav directly.", "warn")))

    # Audio quality analysis
    try:
        data, sr = sf.read(ref_save, always_2d=False)
        dur      = len(data) / sr
        channels = "Stereo" if data.ndim == 2 else "Mono"
        qual     = ("✅ OK (8-30 s ideal range)" if 8 <= dur <= 30
                    else "⚠ VERY SHORT (<8 s) -- extend clip for better cloning" if dur < 8
                    else "⚠ LONG (>30 s) -- trim to 10-20 s for best results")
        peak_val = float(np.abs(data).max())
        ref_rows = [
            ("File",        ref_fname,           ""),
            ("Sample Rate", f"{sr} Hz",          ""),
            ("Duration",    f"{dur:.2f} s",      qual),
            ("Channels",    channels,            ""),
            ("Peak Level",  f"{peak_val:.4f}",
             "✅ OK" if peak_val > 0.05 else "⚠ VERY QUIET -- normalise before use"),
        ]
        display(HTML(_panel("REFERENCE AUDIO ANALYSIS", ref_rows)))
    except Exception as e:
        display(HTML(_alert(f"Could not read audio stats: {e}", "warn")))

    REFERENCE_WAV_PATH = ref_save
    PROMPT_WAV_PATH    = ref_save

    print("\n>>> Paste the EXACT transcript of your reference audio clip.")
    print("    Leave blank to use Voice Clone mode (no transcript required):")
    PROMPT_TEXT = input("Transcript > ").strip() or None

mode = ("Ultimate Clone" if REFERENCE_WAV_PATH and PROMPT_TEXT
        else "Voice Clone"    if REFERENCE_WAV_PATH
        else "Default TTS")

summary = [
    ("Active Mode",  mode,                                          ""),
    ("Audio File",   REFERENCE_WAV_PATH or "None -- Default TTS",  ""),
    ("Transcript",   ((PROMPT_TEXT[:60] + "...") if PROMPT_TEXT and len(PROMPT_TEXT) > 60
                      else PROMPT_TEXT or "None"),                  ""),
]
display(HTML(_panel("CLONING SUMMARY", summary)))
display(HTML(_alert(
    "Reference config confirmed. Proceed to Cell 5 to set TTS parameters.", "success")))


In [ ]:
# ================================================================
# CELL 5 — TTS CONFIGURATION
# ================================================================
# ┌──────────────────────────────────────────────────────────────┐
# │  ════════════════  USER CONFIG  ════════════════════════     │
# │  Edit values in this block ONLY.                            │
# │  All other cells run without changes.                       │
# └──────────────────────────────────────────────────────────────┘

# ── Output ───────────────────────────────────────────────────────
OUTPUT_FILENAME  = "jarvis_audiobook_output"   # filename, no extension
OUTPUT_PATH      = f"/content/{OUTPUT_FILENAME}.wav"

# ── Chunk size (characters per TTS call) ─────────────────────────
# Audiobooks: 350-420 chars is ideal -- longer gives more natural prosody.
# VoxCPM2 safe context limit: <= 480 chars.  Do NOT exceed 480.
# Too short (<150) = choppy joins; too long (>450) = VRAM spikes on T4.
CHUNK_SIZE       = 380   # recommended: 320-420

# ── Minimum chunk size ───────────────────────────────────────────
# Chunks shorter than this are merged with the previous chunk.
# Prevents isolated short sentences producing unnatural audio clips.
MIN_CHUNK_SIZE   = 80    # chars; recommended: 60-100

# ── Inference quality ─────────────────────────────────────────────
# 10 = fast draft,  20 = balanced,  32 = best quality (recommended for commercial)
INFERENCE_TIMESTEPS = 32

# ── CFG strength (style adherence) ────────────────────────────────
# 1.5 = loose / creative,  2.0 = balanced (recommended),  3.0 = strict
CFG_VALUE        = 2.0

# ── Style prefix (VoxCPM2 natural-language voice control) ─────────
# Prepended to every chunk sent to the model.
# For Hinglish modern web-series / novel narration:
STYLE_PREFIX = ""

# ── Silence durations between chunks (seconds) ───────────────────
SILENCE_CHAPTER   = 2.00   # after detected chapter headings / separators
SILENCE_PARAGRAPH = 0.55   # after paragraph boundary
SILENCE_SENTENCE  = 0.20   # after sentence boundary (mid-paragraph)
SILENCE_CLAUSE    = 0.08   # after clause-level split (comma / dash)

# ── Crossfade at chunk tail (milliseconds) ────────────────────────
# Short fade-out applied to the end of each chunk before the silence gap.
# Eliminates click / discontinuity artifacts at stitch points.
# Recommended: 10-20 ms.  Set 0 to disable.
CROSSFADE_MS     = 15

# ── Resume / checkpoint ───────────────────────────────────────────
# RESUME_CHUNKS = True: each chunk is cached to CHUNKS_DIR as .npy.
# On re-run after a VRAM crash, already-completed chunks load from disk
# instantly -- only failed chunks are re-generated.
RESUME_CHUNKS    = True
CHUNKS_DIR       = "/content/chunks"

# ─────────────────────────────────────────────────────────────────
#  END OF USER CONFIG -- do not edit below this line
# ─────────────────────────────────────────────────────────────────
from IPython.display import display, HTML
import os

# ENABLE_NORMALIZATION must have been set in Cell 3; provide a safe default
if "ENABLE_NORMALIZATION" not in dir():
    ENABLE_NORMALIZATION = True

# Clamp / validate all numeric parameters
CHUNK_SIZE          = max(150, min(480, int(CHUNK_SIZE)))
MIN_CHUNK_SIZE      = max(30,  min(200, int(MIN_CHUNK_SIZE)))
INFERENCE_TIMESTEPS = int(INFERENCE_TIMESTEPS) if INFERENCE_TIMESTEPS in (10, 20, 32) else 20
CFG_VALUE           = float(CFG_VALUE) if 0.5 <= float(CFG_VALUE) <= 5.0 else 2.0
CROSSFADE_MS        = max(0, min(50, int(CROSSFADE_MS)))
SILENCE_CHAPTER     = max(0.0, float(SILENCE_CHAPTER))
SILENCE_PARAGRAPH   = max(0.0, float(SILENCE_PARAGRAPH))
SILENCE_SENTENCE    = max(0.0, float(SILENCE_SENTENCE))
SILENCE_CLAUSE      = max(0.0, float(SILENCE_CLAUSE))

rows = [
    ("Output File",     f"{OUTPUT_FILENAME}.wav",                    ""),
    None,
    "CHUNKING",
    ("Chunk Size",      f"{CHUNK_SIZE} chars",                       ""),
    ("Min Chunk Size",  f"{MIN_CHUNK_SIZE} chars",                   ""),
    None,
    "INFERENCE",
    ("Timesteps",       str(INFERENCE_TIMESTEPS),
     "FAST"     if INFERENCE_TIMESTEPS == 10
     else "BALANCED" if INFERENCE_TIMESTEPS == 20
     else "BEST"),
    ("CFG Strength",    str(CFG_VALUE),                              ""),
    ("Style Prefix",    (STYLE_PREFIX[:72] + "...") if len(STYLE_PREFIX) > 72
                        else STYLE_PREFIX or "(none)",               ""),
    None,
    "SILENCE GAPS",
    ("Chapter Break",   f"{SILENCE_CHAPTER*1000:.0f} ms",           ""),
    ("Paragraph Break", f"{SILENCE_PARAGRAPH*1000:.0f} ms",         ""),
    ("Sentence Break",  f"{SILENCE_SENTENCE*1000:.0f} ms",          ""),
    ("Clause Break",    f"{SILENCE_CLAUSE*1000:.0f} ms",            ""),
    None,
    "STITCHING + RESUME",
    ("Crossfade",       f"{CROSSFADE_MS} ms fade-out per chunk",
     "ACTIVE" if CROSSFADE_MS > 0 else "OFF"),
    ("Resume Chunks",   "ENABLED" if RESUME_CHUNKS else "DISABLED",
     "✅ ON" if RESUME_CHUNKS else "⚠ OFF"),
    ("Chunks Dir",      CHUNKS_DIR if RESUME_CHUNKS else "--",       ""),
    None,
    "TEXT PREPROCESSING",
    ("Normalization",
     "ENABLED (Layer 2 ON)" if ENABLE_NORMALIZATION else "DISABLED (raw text)",
     "✅ ON" if ENABLE_NORMALIZATION else "⚠ OFF"),
]

display(HTML(_banner("TTS CONFIGURATION", "PARAMETERS LOCKED IN", "CELL 5 / 7")))
display(HTML(_panel("ACTIVE PARAMETERS", rows,
    note="Edit the USER CONFIG block above and re-run this cell to update any setting.")))
display(HTML(_alert("Parameters confirmed. Proceed to Cell 6 to generate TTS.", "success")))


In [ ]:
# ================================================================
# CELL 6 — LOAD MODEL  +  GENERATE TTS
# ================================================================
import re, time, os, gc
import unicodedata
import numpy as np
import soundfile as sf
import torch
from IPython.display import display, HTML, clear_output
from voxcpm import VoxCPM

# ── Dependency guards ────────────────────────────────────────────
assert "INPUT_TEXT" in globals() and INPUT_TEXT, \
    "Run Cell 3 first -- INPUT_TEXT not defined."
assert "CHUNK_SIZE" in globals(), \
    "Run Cell 5 first -- TTS parameters not set."
assert "REFERENCE_WAV_PATH" in globals(), \
    "Run Cell 4 first -- voice clone config not set."

display(HTML(_banner(
    "TTS GENERATION ENGINE",
    "VoxCPM2  AUDIOBOOK MODE  v3",
    "CELL 6 / 7")))

# ════════════════════════════════════════════════════════════════
#  AUDIOBOOK CHUNKER v3
# ════════════════════════════════════════════════════════════════

# Terminal punctuation marks (Devanagari danda + Latin)
_TERMINAL_CHARS = set("।.!?")

# Split sentence at sentence boundary  (after .!?। followed by whitespace)
_SENT_RE   = re.compile(r"(?<=[।.!?])\s+")

# Split at clause boundary  (comma, semicolon, em-dash followed by whitespace)
_CLAUSE_RE = re.compile(r"(?<=[,;\u2014])\s+")

# Chapter / section heading detector
# Matches: Chapter 1, CHAPTER ONE, अध्याय 2, भाग 3, Part IV,
#          Prologue, Epilogue, lines of ===== or -----, Markdown headings
_CHAPTER_RE = re.compile(
    r"^(?:chapter\s*[\divxlc]+\b"
    r"|chapter\s+\w+\b"
    r"|\u0905\u0927\u094d\u092f\u093e\u092f\s*\d+"
    r"|\u092d\u093e\u0917\s*\d+"
    r"|part\s*[\divxlc]+\b"
    r"|prologue|epilogue|preface|foreword|afterword"
    r"|\u092a\u094d\u0930\u0938\u094d\u0924\u093e\u0935\u0928\u093e"
    r"|\u0909\u092a\u0938\u0902\u0939\u093e\u0930"
    r"|[=\-]{5,}"
    r"|#{1,3}\s+)",
    re.IGNORECASE)

# ── Per-paragraph style cue extractor ─────────────────────────────────────────
# Paragraphs in this text start with a parenthetical style instruction like:
#   (warm engaging narrator, Hinglish flow)actual content...
# When a paragraph is split into multiple chunks, we propagate the style cue to
# ALL resulting chunks so the model receives consistent voice guidance for every
# inference call — not just the first one.
_STYLE_CUE_RE = re.compile(r"^\(([^)\n]{5,250})\)\s*")

def _extract_style_cue(para_text):
    m = _STYLE_CUE_RE.match(para_text.strip())
    if m:
        inner = m.group(1).strip()
        # Accept as a style cue if no sentence-ending punctuation (not regular dialogue)
        if not re.search(r"[\.!?\u0964]", inner):
            cue = "(" + inner + ")"
            rest = para_text.strip()[m.end():]
            return cue, rest
    return None, para_text

# ── Separator chunk detector ────────────────────────────────────────────────
# Catches chunks that are ONLY decorative characters (after terminal punct was
# appended by _ensure_terminal_punct).  These must NOT be sent to the TTS model.
_SEPARATOR_CHUNK_RE = re.compile(r"^[=\-#*]{5,}[.。]?\s*$")


def _ensure_terminal_punct(text):
    t = text.rstrip()
    if not t or t[-1] in _TERMINAL_CHARS:
        return t
    # Devanagari script detected in trailing portion -> use danda
    if re.search(r"[\u0900-\u097F]\s*$", t):
        return t + "\u0964"   # Devanagari danda
    return t + "."


def _hard_split_words(text, n):
    parts, buf = [], ""
    for word in text.split():
        test = (buf + " " + word).strip()
        if len(test) <= n:
            buf = test
        else:
            if buf:
                parts.append(buf)
            buf = word[:n] if len(word) > n else word
    if buf:
        parts.append(buf)
    return parts


def _merge_chunks(items, max_c, min_c=0):
    chunks, buf = [], ""
    for item in items:
        item = item.strip()
        if not item:
            continue
        if len(item) > max_c:
            if buf:
                chunks.append(buf)
                buf = ""
            chunks.extend(_hard_split_words(item, max_c))
        elif len(buf) + len(item) + 1 <= max_c:
            buf = (buf + " " + item).strip()
        else:
            if buf:
                chunks.append(buf)
            buf = item
    if buf:
        chunks.append(buf)

    # Merge tiny trailing chunks upward into previous chunk if it still fits
    merged = []
    for ch in chunks:
        if merged and len(ch) < min_c and len(merged[-1]) + len(ch) + 1 <= max_c:
            merged[-1] = (merged[-1] + " " + ch).strip()
        else:
            merged.append(ch)
    return [c for c in merged if c]


def smart_chunk_audiobook(text, max_chars, min_chars):
    # Break text into paragraphs (double-newline separated)
    paragraphs = [p.strip() for p in re.split(r"\n\n+", text) if p.strip()]
    result     = []
    n_para     = len(paragraphs)

    for p_idx, para in enumerate(paragraphs):
        is_last_para = (p_idx == n_para - 1)

        # Detect chapter / section heading: single line, short, matches pattern
        if ("\n" not in para.strip()
                and len(para) < 120
                and _CHAPTER_RE.match(para.strip())):
            chunk = _ensure_terminal_punct(para.strip())
            btype = "end" if is_last_para else "chapter"
            result.append((chunk, btype))
            continue

        # ── Extract leading style cue, propagate to ALL chunks of paragraph ──
        para_cue, para_content = _extract_style_cue(para)
        # Reduce effective chunk size to leave room for the style cue header
        cue_overhead  = len(para_cue) + 1 if para_cue else 0
        effective_max = max(80, max_chars - cue_overhead)

        # Sentence-level split on the CONTENT portion (cue already removed)
        sentences = [s.strip() for s in _SENT_RE.split(para_content) if s.strip()]

        expanded = []
        for sent in sentences:
            if len(sent) <= effective_max:
                expanded.append(sent)
            else:
                # Try clause split
                clauses = [c.strip() for c in _CLAUSE_RE.split(sent) if c.strip()]
                if len(clauses) > 1:
                    expanded.extend(_merge_chunks(clauses, effective_max, 0))
                else:
                    # Last resort: word-boundary split
                    expanded.extend(_hard_split_words(sent, effective_max))

        para_chunks = _merge_chunks(expanded, effective_max, min_chars)
        n_pc        = len(para_chunks)

        for c_idx, chunk in enumerate(para_chunks):
            is_last_chunk = (c_idx == n_pc - 1)
            chunk = _ensure_terminal_punct(chunk)
            # Re-attach style cue to every chunk from this paragraph
            if para_cue:
                chunk = para_cue + chunk
            if is_last_chunk and is_last_para:
                btype = "end"
            elif is_last_chunk:
                btype = "paragraph"
            else:
                btype = "sentence"
            result.append((chunk, btype))

    return result


# ════════════════════════════════════════════════════════════════
#  CROSSFADE STITCHER
# ════════════════════════════════════════════════════════════════

def stitch_audio(segments, gap_types, silence_map, sr, fade_ms):
    fade_n = max(0, int(fade_ms * sr / 1000))
    pieces = []
    if fade_n > 0:
        fade_curve = np.linspace(1.0, 0.0, fade_n, dtype=np.float32)

    for i, seg in enumerate(segments):
        seg_arr = np.asarray(seg, dtype=np.float32)

        if seg_arr.size > 0:
            # Normal audio segment: apply fade-out to tail then append
            if fade_n > 0 and len(seg_arr) > fade_n * 3:
                seg_arr = seg_arr.copy()
                seg_arr[-fade_n:] *= fade_curve
            pieces.append(seg_arr)
        # If seg_arr.size == 0: pure separator — no audio, only the gap below

        # Always insert the configured gap (silence) after each segment
        if i < len(gap_types):
            sil_sec = silence_map.get(gap_types[i], 0.0)
            if sil_sec > 0:
                pieces.append(np.zeros(int(sil_sec * sr), dtype=np.float32))

    if not pieces:
        return np.array([], dtype=np.float32)
    return np.concatenate(pieces).astype(np.float32)


# ════════════════════════════════════════════════════════════════
#  STEP A -- CHUNKING
# ════════════════════════════════════════════════════════════════
print("Chunking text ...")
chunk_pairs  = smart_chunk_audiobook(INPUT_TEXT, CHUNK_SIZE, MIN_CHUNK_SIZE)
total_chunks = len(chunk_pairs)
total_chars  = sum(len(c) for c, _ in chunk_pairs)
break_counts = {}
for _, b in chunk_pairs:
    break_counts[b] = break_counts.get(b, 0) + 1

chunk_rows = [
    ("Total Chunks",     str(total_chunks),                                  ""),
    ("Total Characters", f"{total_chars:,}",                                ""),
    ("Avg Chunk Size",   f"{total_chars // max(total_chunks, 1)} chars",    ""),
    None,
    "BREAK DISTRIBUTION",
] + [(f"  {k} breaks", str(v), "") for k, v in sorted(break_counts.items())]
display(HTML(_panel("CHUNKING SUMMARY", chunk_rows, accent="#f39c12")))

preview_rows = []
for i, (c, bt) in enumerate(chunk_pairs[:5], 1):
    preview_rows.append((
        f"Chunk {i:03d}  [{bt}]",
        c[:80] + ("..." if len(c) > 80 else ""), ""))
if total_chunks > 5:
    preview_rows.append(("...", f"and {total_chunks - 5} more chunks", ""))
display(HTML(_panel("CHUNK PREVIEW (first 5)", preview_rows, accent="#f39c12")))


# ════════════════════════════════════════════════════════════════
#  STEP B -- LOAD MODEL
# ════════════════════════════════════════════════════════════════
print("\nLoading VoxCPM2 model (~8 GB download on first run) ...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

t_load = time.time()
# T4 has 16 GB VRAM; VoxCPM2 + denoiser needs ~7-8 GB — well within budget.
# The denoiser significantly improves audio naturalness for commercial use.
model  = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=True)
load_time = time.time() - t_load

# Detect sample rate robustly
SAMPLE_RATE = None
for _attr_path in ("tts_model.sample_rate", "sample_rate"):
    try:
        obj = model
        for part in _attr_path.split("."):
            obj = getattr(obj, part)
        SAMPLE_RATE = int(obj)
        break
    except AttributeError:
        pass
if SAMPLE_RATE is None:
    SAMPLE_RATE = 48_000
    display(HTML(_alert(
        "sample_rate attribute not found on model -- defaulting to 48 000 Hz.", "warn")))

model_rows = [
    ("Model",       "openbmb/VoxCPM2",        ""),
    ("Load Time",   f"{load_time:.1f} s",      "✅ OK"),
    ("Sample Rate", f"{SAMPLE_RATE:,} Hz",     ""),
]
if torch.cuda.is_available():
    used_gb  = torch.cuda.memory_allocated() / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    model_rows.append((
        "VRAM Used",
        f"{used_gb:.2f} GB / {total_gb:.2f} GB",
        "✅ OK" if used_gb < total_gb * 0.85 else "⚠ HIGH"))
display(HTML(_panel("MODEL STATUS", model_rows, accent="#2ecc71")))


# ════════════════════════════════════════════════════════════════
#  STEP C -- GENERATE CHUNK BY CHUNK
# ════════════════════════════════════════════════════════════════
if RESUME_CHUNKS:
    os.makedirs(CHUNKS_DIR, exist_ok=True)

SILENCE_MAP = {
    "chapter":   SILENCE_CHAPTER,
    "paragraph": SILENCE_PARAGRAPH,
    "sentence":  SILENCE_SENTENCE,
    "clause":    SILENCE_CLAUSE,
    "end":       0.0,
}

mode_label = ("Ultimate Clone" if (REFERENCE_WAV_PATH and PROMPT_TEXT)
              else "Voice Clone" if REFERENCE_WAV_PATH
              else "Default TTS")

all_audio    = []
chunk_breaks = []
chunk_times  = []
failed_idx   = []
resumed_idx  = []
gen_start    = time.time()


def _render_progress(done, total, ctimes, failed, resumed, eta_str, current, btype):
    pct   = (done / total) * 100 if total else 0
    bar_f = int(pct / 100 * 28)
    bar   = chr(0x2588) * bar_f + chr(0x2591) * (28 - bar_f)
    # Filter to real generation times only (exclude resumed=0.0 placeholders).
    # resumed contains 1-based chunk indices; ctimes is 0-indexed → offset by 1.
    active_t = [t for i, t in enumerate(ctimes) if (i + 1) not in set(resumed)]
    avg = sum(active_t) / len(active_t) if active_t else 0
    rows = [
        ("Progress",      f"[{bar}]  {pct:.1f}%  ({done}/{total})",         ""),
        ("Current Chunk", f"{current[:70]}{'...' if len(current)>70 else ''}", ""),
        ("Break After",   btype,                                              ""),
        ("Avg / Chunk",   f"{avg:.1f} s" if avg else "--",                   ""),
        ("ETA",           eta_str,                                            ""),
        ("Resumed",       str(len(resumed)) if resumed else "None",           ""),
        ("Failed",        str(len(failed))  if failed  else "None",
         "" if not failed else "⚠ CHECK"),
    ]
    return (_panel("TTS GENERATION PROGRESS", rows, accent="#e74c3c", width="700px")
            + _pbar(pct, color="#e74c3c"))


for idx, (chunk, btype) in enumerate(chunk_pairs, 1):
    chunk_path = (os.path.join(CHUNKS_DIR, f"chunk_{idx:04d}.npy")
                  if RESUME_CHUNKS else None)

    # ── Pure separator chunk: no audio, just carry the gap silence ───────────
    # Separator lines (=====) are detected here after _ensure_terminal_punct
    # may have appended a dot. Skip TTS entirely — stitch_audio handles them.
    if _SEPARATOR_CHUNK_RE.match(chunk.strip()):
        all_audio.append(np.array([], dtype=np.float32))
        chunk_breaks.append(btype)
        chunk_times.append(0.0)
        continue

    # Resume: load from cache if available and valid
    if RESUME_CHUNKS and chunk_path and os.path.exists(chunk_path):
        try:
            wav = np.load(chunk_path)
            if wav.size > 0:
                all_audio.append(wav.astype(np.float32))
                chunk_breaks.append(btype)
                chunk_times.append(0.0)
                resumed_idx.append(idx)
                continue
        except Exception:
            pass   # corrupt cache -> regenerate

    t_chunk = time.time()

    # ETA (use only real generation times, not resume placeholders)
    real_times = [t for i, t in enumerate(chunk_times)
                  if (i + 1) not in set(resumed_idx) and t > 0]
    if real_times:
        avg_t   = sum(real_times) / len(real_times)
        eta_sec = avg_t * (total_chunks - idx + 1)
        eta_str = f"{eta_sec/60:.1f} min" if eta_sec > 60 else f"{eta_sec:.0f} s"
    else:
        eta_str = "calculating..."

    clear_output(wait=True)
    display(HTML(_banner("TTS GENERATION", f"Mode: {mode_label}", "CELL 6 / 7")))
    display(HTML(_render_progress(
        idx - 1, total_chunks, chunk_times,
        failed_idx, resumed_idx, eta_str, chunk, btype)))

    # Prepend style prefix (adds voice guidance context)
    chunk_text = (STYLE_PREFIX + chunk) if STYLE_PREFIX else chunk

    gen_kwargs = dict(
        text=chunk_text,
        cfg_value=CFG_VALUE,
        inference_timesteps=INFERENCE_TIMESTEPS,
    )
    if REFERENCE_WAV_PATH:
        gen_kwargs["reference_wav_path"] = REFERENCE_WAV_PATH
    if PROMPT_WAV_PATH and PROMPT_TEXT:
        gen_kwargs["prompt_wav_path"] = PROMPT_WAV_PATH
        gen_kwargs["prompt_text"]     = PROMPT_TEXT

    # Generate with up to 3 attempts (handles transient VRAM / model errors)
    wav = None
    for attempt in range(3):
        try:
            wav = model.generate(**gen_kwargs)
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(4)
            if attempt == 2:
                display(HTML(_alert(
                    f"Chunk {idx}: VRAM OOM after 3 attempts -- skipped. "
                    "Re-run with RESUME_CHUNKS=True to retry.", "error")))
        except Exception as e:
            if attempt < 2:
                time.sleep(2)
                continue
            display(HTML(_alert(f"Chunk {idx} failed after 3 attempts: {e}", "error")))

    if wav is None:
        failed_idx.append(idx)
        continue

    # Shape normalisation: ensure 1-D float32
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=0)   # stereo -> mono

    if wav.size == 0:
        failed_idx.append(idx)
        continue

    # Hard clamp to [-1, 1] to prevent overflow in later normalisation
    wav = np.clip(wav, -1.0, 1.0)

    # Persist to disk for resume capability
    if RESUME_CHUNKS and chunk_path:
        try:
            np.save(chunk_path, wav)
        except Exception as e:
            display(HTML(_alert(f"Cache write failed chunk {idx}: {e}", "warn")))

    all_audio.append(wav)
    chunk_breaks.append(btype)
    chunk_times.append(time.time() - t_chunk)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ════════════════════════════════════════════════════════════════
#  STEP D -- STITCH + SAVE
# ════════════════════════════════════════════════════════════════
clear_output(wait=True)
display(HTML(_banner("GENERATION COMPLETE", "STITCHING AND SAVING AUDIO", "CELL 6 / 7")))

if not all_audio:
    raise RuntimeError(
        "No audio segments were generated. Check the errors printed above.")

print(f"Stitching {len(all_audio)} segments (crossfade={CROSSFADE_MS} ms) ...")
final_audio = stitch_audio(
    all_audio, chunk_breaks, SILENCE_MAP, SAMPLE_RATE, CROSSFADE_MS)

# Peak-normalise to -1 dBFS
peak = float(np.abs(final_audio).max())
if peak > 0:
    target = 10 ** (-1.0 / 20)   # -1 dBFS
    final_audio = (final_audio / peak * target).astype(np.float32)

sf.write(OUTPUT_PATH, final_audio, SAMPLE_RATE, subtype="PCM_16")

total_time = time.time() - gen_start
audio_dur  = len(final_audio) / SAMPLE_RATE
file_mb    = os.path.getsize(OUTPUT_PATH) / 1e6
real_times = [t for i, t in enumerate(chunk_times)
              if (i + 1) not in set(resumed_idx) and t > 0]
avg_ct     = sum(real_times) / len(real_times) if real_times else 0
rtf        = total_time / audio_dur if audio_dur > 0 else 0

final_rows = [
    ("Chunks OK / Total",  f"{len(all_audio)} / {total_chunks}",
     "✅ OK" if not failed_idx else f"⚠ {len(failed_idx)} failed"),
    ("Resumed from Cache", str(len(resumed_idx)) if resumed_idx else "None", ""),
    ("Total Wall Time",    f"{total_time/60:.1f} min  ({total_time:.0f} s)", ""),
    ("Avg Gen / Chunk",    f"{avg_ct:.1f} s" if avg_ct else "N/A (all resumed)", ""),
    ("Audio Duration",     f"{audio_dur:.1f} s  ({audio_dur/60:.1f} min)",   ""),
    ("Real-Time Factor",   f"{rtf:.2f}x",                                     ""),
    ("Sample Rate",        f"{SAMPLE_RATE:,} Hz",                             ""),
    ("Output File Size",   f"{file_mb:.2f} MB",                              ""),
    ("Saved To",           OUTPUT_PATH,                                       "✅ WRITTEN"),
]
display(HTML(_panel("FINAL REPORT", final_rows, accent="#2ecc71",
    note="Proceed to Cell 7 to play back and download your audiobook.")))

if failed_idx:
    display(HTML(_alert(
        f"Failed chunks: {failed_idx}  --  re-run Cell 6 with RESUME_CHUNKS=True "
        "to regenerate only these.", "warn")))

display(HTML(_alert(
    "ALL SYSTEMS GREEN  MISSION SUCCESS  Proceed to Cell 7.", "success")))


In [ ]:
# ================================================================
# CELL 7 — PLAYBACK & DOWNLOAD
# ================================================================
from google.colab import files
from IPython.display import Audio, display, HTML
import soundfile as sf, os, numpy as np

display(HTML(_banner("PLAYBACK & DOWNLOAD", "JARVIS SESSION FINALIZING", "CELL 7 / 7")))

if not os.path.exists(OUTPUT_PATH):
    raise FileNotFoundError(
        f"Output file not found: {OUTPUT_PATH}\n"
        "Run Cell 6 first to generate the audiobook.")

data, sr = sf.read(OUTPUT_PATH, always_2d=False)
dur      = len(data) / sr
size_mb  = os.path.getsize(OUTPUT_PATH) / 1e6
peak_val = float(np.abs(data).max())
peak_db  = 20 * np.log10(peak_val) if peak_val > 0 else float("-inf")

rows = [
    ("Filename",    f"{OUTPUT_FILENAME}.wav",              ""),
    ("Duration",    f"{dur:.1f} s  ({dur/60:.1f} min)",   ""),
    ("Sample Rate", f"{sr:,} Hz",                          ""),
    ("File Size",   f"{size_mb:.2f} MB",                  ""),
    ("Peak Level",  f"{peak_db:.2f} dBFS",
     "✅ OK" if -3.0 <= peak_db <= 0.0 else "⚠ CHECK LEVEL"),
]
display(HTML(_panel("OUTPUT FILE", rows, accent="#2ecc71")))

print("\nIN-NOTEBOOK PLAYBACK:")
display(Audio(OUTPUT_PATH, autoplay=False))

print("\nInitiating download to your local machine ...")
files.download(OUTPUT_PATH)

display(HTML(_alert(
    f"Download initiated -- check your browser for  {OUTPUT_FILENAME}.wav", "success")))

display(HTML(
    '<div style="background:#060606;border:1.5px solid #2ecc71;border-radius:8px;'
    'padding:16px 22px;margin:12px 0;font-family:Courier New,monospace;'
    'max-width:700px;box-shadow:0 0 18px #2ecc7122;text-align:center">'
    '<div style="color:#2ecc71;font-size:15px;font-weight:bold;letter-spacing:3px">'
    '&#x2B21; JARVIS SESSION COMPLETE &#x2B21;</div>'
    '<div style="color:#555;font-size:10px;letter-spacing:2px;margin-top:6px">'
    'READLYTE PRODUCTION  VoxCPM2 v3  HINGLISH AUDIOBOOK OUTPUT READY</div>'
    '</div>'
))
